# Round 5 Frozen Pipeline

This notebook calls the shared Hoffman2 array and report scripts. It does not duplicate model or analysis code.

In [ ]:
# Run once per fresh notebook environment.
%pip install -q numpy pandas matplotlib

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / 'r5_array_workflow.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
print(PROJECT_ROOT)

In [ ]:
# Frozen run settings. Start with a tiny smoke configuration before a full run.
FAMILY = 'six_sample'  # oracle, six_sample, custom_rr, or fixed_budget
OUTPUT_DIR = 'results/r5_notebook_run'
EPISODES = 2
EPISODES_PER_TASK = 1
VOI_SAMPLES = 4
SEED_NAMESPACE_OFFSET = 0
MAX_CONCURRENT = 2
CONFIGS_JSON = ''  # Required for custom_rr and fixed_budget.
RR_POLICY = 'myopic_voi'  # myopic_voi or discretized_dp
SUBMIT_TO_HOFFMAN2 = False

In [ ]:
environment = os.environ.copy()
environment.update({
    'FAMILY': FAMILY,
    'OUTPUT_DIR': OUTPUT_DIR,
    'EPISODES': str(EPISODES),
    'EPISODES_PER_TASK': str(EPISODES_PER_TASK),
    'OBSERVATION_DRAWS': str(VOI_SAMPLES),
    'SEED_NAMESPACE_OFFSET': str(SEED_NAMESPACE_OFFSET),
    'MAX_CONCURRENT': str(MAX_CONCURRENT),
    'CONFIGS_JSON': CONFIGS_JSON,
    'RR_POLICY': RR_POLICY,
})
submit_command = ['bash', str(PROJECT_ROOT / 'scripts' / 'submit_hoffman2_round5_array.sh')]
print(' '.join(f'{key}={environment[key]}' for key in [
    'FAMILY', 'OUTPUT_DIR', 'EPISODES', 'EPISODES_PER_TASK',
    'OBSERVATION_DRAWS', 'SEED_NAMESPACE_OFFSET', 'MAX_CONCURRENT',
]) + ' ' + ' '.join(submit_command))
if SUBMIT_TO_HOFFMAN2:
    module_init = Path('/u/local/Modules/default/init/bash')
    if not module_init.exists():
        raise RuntimeError('Submit this array from Hoffman2, not from a local notebook.')
    subprocess.run(submit_command, cwd=PROJECT_ROOT, env=environment, check=True)

In [ ]:
# This reads atomic task statuses; it never reruns missing simulations.
manifest = PROJECT_ROOT / OUTPUT_DIR / 'r5_manifest.json'
if manifest.exists():
    subprocess.run([
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'r5_array_workflow.py'),
        'progress', '--manifest', str(manifest),
    ], cwd=PROJECT_ROOT, check=True)
else:
    print(f'Manifest not found yet: {manifest}')

In [ ]:
# Generate the final report only after every input has a strict collector output.
GENERATE_REPORT = False
REPORT_INPUTS = {
    'oracle-dir': 'results/r5_oracle_full',
    'oracle-analysis-dir': 'results/r5_oracle_analysis',
    'formal-dir': 'results/r5_formal_summaries',
    'discovery-dir': 'results/r5_six_sample_discovery',
    'confirmation-dir': 'results/r5_six_sample_confirmation',
    'solver-dir': 'results/r5_solver_comparison',
    'output-dir': 'results/round5_report',
}
report_command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'analyze_round5.py')]
for option, value in REPORT_INPUTS.items():
    report_command.extend([f'--{option}', str(PROJECT_ROOT / value)])
print(' '.join(report_command))
if GENERATE_REPORT:
    subprocess.run(report_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
import pandas as pd

report_dir = PROJECT_ROOT / REPORT_INPUTS['output-dir']
summary_path = report_dir / 'supporting_data' / 'r5_report_summary.json'
confirmation_path = report_dir / 'supporting_data' / 'r5_confirmation_comparison.csv'
if summary_path.exists():
    display(json.loads(summary_path.read_text(encoding='utf-8')))
if confirmation_path.exists():
    display(pd.read_csv(confirmation_path))